# Generate Datasets for All Configs

This notebook generates syndrome data for all 12 (d, p) configurations.
- Distances: d ∈ {3, 5, 7}
- Physical error rates: p ∈ {0.001, 0.005, 0.01, 0.05}
- Samples per config: 1,000,000
- Rounds: 2

Each config is saved as a .npz file containing: measurements, det_evts, flips

In [11]:
import numpy as np
import stim
import os
from circuit_generators import get_builtin_circuit
from datetime import datetime

In [7]:
print(f'Starting dataset generation at {datetime.now()}')

# Configuration
CONFIGS = [
    (3, 0.001), (3, 0.005), (3, 0.010), (3, 0.050),
    (5, 0.001), (5, 0.005), (5, 0.010), (5, 0.050),
    (7, 0.001), (7, 0.005), (7, 0.010), (7, 0.050),
]
N_SAMPLES = 1_000_000
ROUNDS = 2
OUTPUT_DIR = './datasets'
SEED = 42

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output directory: {OUTPUT_DIR}')
print(f'Total configs: {len(CONFIGS)}')
print(f'Samples per config: {N_SAMPLES:,}')
print(f'Total samples: {len(CONFIGS) * N_SAMPLES:,}\n')

Starting dataset generation at 2026-05-29 00:54:23.753161
Output directory: ./datasets
Total configs: 12
Samples per config: 1,000,000
Total samples: 12,000,000



In [8]:
# Generate data for each config
for i, (d, p) in enumerate(CONFIGS, 1):
    print(f'[{i}/{len(CONFIGS)}] Generating d={d}, p={p:.3f}...', end=' ', flush=True)
    
    try:
        # Generate circuit
        circuit = get_builtin_circuit(
            'surface_code:rotated_memory_z',
            distance=d,
            rounds=ROUNDS,
            before_round_data_depolarization=p,
            after_reset_flip_probability=p,
            after_clifford_depolarization=p,
        )
        
        # Sample measurements
        m_sampler = circuit.compile_sampler(seed=SEED)
        measurements = m_sampler.sample(N_SAMPLES, bit_packed=False)
        
        # Convert to detector events and observables
        converter = circuit.compile_m2d_converter()
        det_evts, flips = converter.convert(
            measurements=measurements,
            separate_observables=True,
            bit_packed=False
        )
        
        # Save to .npz file
        filename = f'{OUTPUT_DIR}/data_d{d}_p{p:.3f}_r{ROUNDS}.npz'
        np.savez(
            filename,
            measurements=measurements,
            det_evts=det_evts,
            flips=flips
        )
        
        print('✓')
        
    except Exception as e:
        print(f'✗ {e}')
        continue
    
    print()

[1/12] Generating d=3, p=0.001... ✓

[2/12] Generating d=3, p=0.005... ✓

[3/12] Generating d=3, p=0.010... ✓

[4/12] Generating d=3, p=0.050... ✓

[5/12] Generating d=5, p=0.001... ✓

[6/12] Generating d=5, p=0.005... ✓

[7/12] Generating d=5, p=0.010... ✓

[8/12] Generating d=5, p=0.050... ✓

[9/12] Generating d=7, p=0.001... ✓

[10/12] Generating d=7, p=0.005... ✓

[11/12] Generating d=7, p=0.010... ✓

[12/12] Generating d=7, p=0.050... ✓



In [9]:
# Verify all files were created
separator = '='*60
print(f'\n{separator}')
print('Dataset generation complete')
print(separator)

files = sorted([f for f in os.listdir(OUTPUT_DIR) if f.endswith('.npz')])
print(f'\nGenerated {len(files)} files:')
for f in files:
    filepath = os.path.join(OUTPUT_DIR, f)
    size_mb = os.path.getsize(filepath) / (1024**2)
    print(f'  {f:40s} ({size_mb:.1f} MB)')

total_size_mb = sum(os.path.getsize(os.path.join(OUTPUT_DIR, f)) for f in files) / (1024**2)
print(f'\nTotal size: {total_size_mb:.1f} MB')
print(f'Completed at {datetime.now()}')


Dataset generation complete

Generated 12 files:
  data_d3_p0.001_r2.npz                    (40.1 MB)
  data_d3_p0.005_r2.npz                    (40.1 MB)
  data_d3_p0.010_r2.npz                    (40.1 MB)
  data_d3_p0.050_r2.npz                    (40.1 MB)
  data_d5_p0.001_r2.npz                    (116.3 MB)
  data_d5_p0.005_r2.npz                    (116.3 MB)
  data_d5_p0.010_r2.npz                    (116.3 MB)
  data_d5_p0.050_r2.npz                    (116.3 MB)
  data_d7_p0.001_r2.npz                    (230.8 MB)
  data_d7_p0.005_r2.npz                    (230.8 MB)
  data_d7_p0.010_r2.npz                    (230.8 MB)
  data_d7_p0.050_r2.npz                    (230.8 MB)

Total size: 1548.8 MB
Completed at 2026-05-29 00:54:37.511567
